# Detect bursts in preprocessed recordings

This example takes the cleaned `.npz` files produced by the preprocessing pipeline (see
`preprocess_pipeline.ipynb`) and runs **burst detection + feature extraction** over them, writing one
burst CSV per recording and a per-experiment burst log.

The moving parts:

- **`Config`** (`mxtreme.config`) — reads `mxtreme.toml` to find the *managed data store*. Inputs are
  read from `config.preprocessed_dir`; burst outputs go to `config.burst_data_dir`.
- **`Recording`** (`mxtreme.recording`) — a thin, experiment-agnostic view over one `.npz`.
- **`Phases`** (`mxtreme.phases`) — optional, *injected* phase labels anchored to maxlab event tags. No
  phases → a single `"full"` phase.
- **`BurstDetector` / `BurstSet`** (`mxtreme.bursting`) — detection (`method="isi_rate"`) and per-burst
  features, parameterized by `BurstDetectParams` / `BurstFeatureParams` (`mxtreme.params`).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import pandas as pd

from mxtreme.config import Config
from mxtreme import io
from mxtreme.recording import Recording
from mxtreme.bursting import BurstDetector
from mxtreme.params import BurstDetectParams, BurstFeatureParams
from mxtreme.phases import phases_from_event_tags

# Managed store from the TOML — inputs read from preprocessed_dir, burst outputs go to burst_data_dir.
config = Config.from_toml("mxtreme.toml")
print("preprocessed dir:", config.preprocessed_dir.resolve())
print("burst_data dir:  ", config.burst_data_dir.resolve())

## 1. Find the preprocessed recordings

Each cleaned recording is a `DIV*_exp_data.npz` under
`<store>/preprocessed/<exp_id>/<chip>/well<well>/`.

In [ ]:
npz_paths = sorted(config.preprocessed_dir.glob("*/*/*/DIV*_exp_data.npz"))
print(f"found {len(npz_paths)} recordings")
for p in npz_paths:
    print(" ", p.relative_to(config.preprocessed_dir))

## 2. Choose detection parameters and phase tags

`BurstDetectParams` / `BurstFeatureParams` carry the knobs (defaults shown). `PHASE_TAGS` is an
**ordered** spec mapping phase names to the maxlab event tags that start them; these `burstTrainer`
recordings tag `pre_recording_start` / `closed_loop_start` / `post_recording_start`. Applied per
recording, tags that aren't present are skipped, and a recording with no tags falls back to a single
`"full"` phase.

In [ ]:
detect_params = BurstDetectParams()      # n=300, noise_thresh=0.05, burst_thresh=0.15, min_dist_bins=30
feature_params = BurstFeatureParams()    # onset_thresh_pct=0.1, offset_thresh_pct=0.05

PHASE_TAGS = [
    ("pre",   "pre_recording_start"),
    ("train", "closed_loop_start"),
    ("post",  "post_recording_start"),
]
END_TAG = "end_experiment"

print(detect_params)
print(feature_params)

## 3. Run detection + feature extraction over every recording

For each npz: load → build phases from its event tags → `Recording(..., phases=phases)` → detect →
extract features → save the burst CSV and update the per-experiment burst log.

> Runtime is dominated by fitting the ISI-N threshold; even a long, dense recording (millions of
> spikes) detects in ~15 s, with feature extraction a couple of seconds more.

In [ ]:
summary = []
for npz in npz_paths:
    data = io.load_preprocessed(npz)

    # Provisional recording to read event tags, then build phases and rebuild with them injected.
    rec = Recording(0, data)
    end_frame = int(rec.spike_data["frameno"].max())
    phases = phases_from_event_tags(rec.event_df, PHASE_TAGS, end_frame=end_frame, end_tag=END_TAG)
    rec = Recording(0, data, phases=phases)

    bursts = BurstDetector(detect_params).detect(rec)
    bursts.extract_features(rec, feature_params)

    csv_path = io.save_burst_data(bursts, config.burst_data_dir, rec)
    io.update_burst_log(config.burst_data_dir, rec, bursts)

    df = bursts.to_dataframe()
    summary.append({
        "recording": f"{rec.exp_id}/{rec.chip}/well{rec.well}/DIV{rec.DIV}",
        "phases": ",".join(phases.names),
        "n_bursts": len(bursts),
        "n_network": int((df["kind"] == "network").sum()),
        "n_mini": int((df["kind"] == "mini").sum()),
        "n_ignored": bursts.n_ignored,
    })

pd.DataFrame(summary)

## 4. Inspect one recording's bursts

Reload a saved CSV with `io.load_burst_data`. Every temporal column is in **frames** (convert with
`mxtreme.utils.frame_to_sec` / `frame_to_bin` as needed); `kind` is `network` or `mini`; `phase` is the
injected label.

In [ ]:
one_csv = io._burst_csv_path(config.burst_data_dir, Recording(0, io.load_preprocessed(npz_paths[0])))
bd = io.load_burst_data(one_csv).to_dataframe()
print(one_csv.name)
print("kind counts:", bd["kind"].value_counts().to_dict())
print("phase counts:", bd["phase"].value_counts(dropna=False).to_dict())
bd[["id", "peak_frame", "peak_amp", "kind", "phase", "onset_frame", "offset_frame",
    "duration_frames", "size_frac_elec", "origin_x", "origin_y", "peak_x", "peak_y"]].head(10)

## 5. Visualize

- **ASDR with bursts** — array-wide rate over a window with network-burst peaks marked.
- **Origin heatmap** — where network bursts originate on the MEA.

In [ ]:
import matplotlib.pyplot as plt
from mxtreme import visualizations as viz

rec = Recording(0, io.load_preprocessed(npz_paths[0]))
burst_df = io.load_burst_data(io._burst_csv_path(config.burst_data_dir, rec)).to_dataframe()

fig, (ax_asdr, ax_mea) = plt.subplots(1, 2, figsize=(16, 5))
viz.plot_bursts_on_asdr(rec, burst_df, ax=ax_asdr, zoom=(0, 6000), title="ASDR with network bursts")
viz.plot_origin_heatmap(rec, burst_df, ax=ax_mea, title="Network-burst origins")
plt.tight_layout()
plt.show()

## 7. Zero-config phases

Injecting no phases is fully supported: every burst is labelled `"full"`. This is the default for any
recording without meaningful event-tag phases.

In [ ]:
rec = Recording(0, io.load_preprocessed(npz_paths[0]))
csv = io._burst_csv_path(config.burst_data_dir, rec)

# Load only the saved detection results (a BurstSet) and recompute features from the npz — no re-detection.
bursts = io.load_burst_data(csv)
bursts.extract_features(rec, feature_params)

io.save_burst_data(bursts, config.burst_data_dir, rec)
log_path = io.update_burst_log(config.burst_data_dir, rec, bursts)

# detection_completed_at is preserved from the earlier run; features_computed_at is refreshed.
log = pd.read_csv(log_path)
row = log[(log.chip == rec.chip) & (log.well == rec.well) & (log.DIV == rec.DIV)].iloc[0]
row[["n_bursts", "n_ignored", "detection_completed_at", "features_computed_at"]]

## 6. Zero-config phases

Injecting no phases is fully supported: every burst is labelled `"full"`. This is the default for any
recording without meaningful event-tag phases.

In [ ]:
rec_full = Recording(0, io.load_preprocessed(npz_paths[0]))   # no phases=
bursts_full = BurstDetector(detect_params).detect(rec_full)
print("phases:", rec_full.phases.names)
print("burst phase labels:", set(b.phase for b in bursts_full))